In [ ]:
!pip install google-generativeai pydantic -q

In [ ]:
import google.generativeai as genai
from google.colab import drive
from pydantic import BaseModel, Field
import json
from IPython.display import display
from PIL import Image
import os


drive.mount('/content/drive')


api_key = "Enter key"
genai.configure(api_key=api_key)
print("Gemini API Key configured successfully!")

In [ ]:
class RelationshipData(BaseModel):
    subject: str = Field(description="The primary person being discussed (e.g., Veena Nayar)")
    related_person: str = Field(description="The person related to the subject, if any. Use 'None' if NA.")
    relationship_type: str = Field(description="How they are related (e.g., Daughter, Son, Daughter-in-law, Grandchild)")
    event: str = Field(description="The main event mentioned (e.g., Death, Chautha & Uthala)")
    event_date: str = Field(description="Date of the event. Return 'Unknown' if not found.")
    event_venue: str = Field(description="The name of the building or venue. Return 'Unknown' if not found.")
    event_address: str = Field(description="The street address and city for the event. Return 'Unknown' if not found.")

def process_image_directly(image_path):
    # Using the current 3.8 Flash model
    model = genai.GenerativeModel('gemini-3.8-flash')

    img = Image.open(image_path)

    prompt = """
    You are an expert information extraction system.
    Read this newspaper clipping carefully. Extract all listed family members and their exact relationship to the deceased.
    Also, extract the venue name and the full address where the scheduled event is taking place.
    Ensure you capture names accurately (e.g., Bobby, Naveen, Ravitha, Mehul, Devaansh).
    """

    response = model.generate_content(
        [prompt, img],
        generation_config=genai.GenerationConfig(
            response_mime_type="application/json",
            response_schema=list[RelationshipData],
        ),
    )

    return json.loads(response.text)

In [ ]:
IMAGE_PATH = '/content/drive/MyDrive/Web3/Photo from Rohit'

if not os.path.exists(IMAGE_PATH):
    print(f" File not found at {IMAGE_PATH}. Please check the path and try again.")
else:
    print(f"\n Processing: {IMAGE_PATH} \n")


    display(Image.open(IMAGE_PATH))


    print("\n Running Gemini Vision Extraction...")
    extracted_data = process_image_directly(IMAGE_PATH)

    print("\n Final Extracted Information (JSON):")
    print(json.dumps(extracted_data, indent=4))